In [1]:
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import threading
import queue
import os
import sys
from datetime import datetime
import time

# ==============================================================================
# BUSINESS LOGIC LAYER
# ==============================================================================

class DataProcessor:
    """
    Handles all data ingestion, transformation, and business logic execution.
    This class is designed to run in a separate thread.
    """

    def load_messy_excel(self, file_path: str, sheet_index: int, unique_header: str = "ACCOUNT NO.") -> pd.DataFrame:
        """
        Loads an Excel sheet by searching for a specific header row and reading
        data until the first blank row in that specific column.

        Args:
            file_path: Path to the Excel file.
            sheet_index: Index of the sheet to load (0-based).
            unique_header: The specific text to look for to identify the header row.

        Returns:
            pd.DataFrame: Cleaned dataframe.
        """
        # Read the raw sheet without headers to scan for structure
        try:
            raw_df = pd.read_excel(file_path, sheet_name=sheet_index, header=None)
        except Exception as e:
            raise ValueError(f"Could not read sheet index {sheet_index}: {e}")

        # 1. Find the row index containing the header
        header_row_idx = None
        for idx, row in raw_df.iterrows():
            # Check if unique_header is present in the row's values (converted to string)
            row_values = [str(x).strip() for x in row.values]
            if unique_header in row_values:
                header_row_idx = idx
                break

        if header_row_idx is None:
            raise ValueError(f"Could not find header '{unique_header}' in sheet {sheet_index}")

        # 2. Reload dataframe using the found header row
        df = pd.read_excel(file_path, sheet_name=sheet_index, header=header_row_idx)

        # 3. Find the first blank row in the unique_header column to stop reading
        # We slice from the beginning.
        valid_indices = []
        
        # Ensure column name matches exactly (strip whitespace just in case)
        df.columns = [str(c).strip() for c in df.columns]
        
        if unique_header not in df.columns:
             raise ValueError(f"Column '{unique_header}' not found after setting header.")

        for idx, value in df[unique_header].items():
            if pd.isna(value) or str(value).strip() == "":
                break
            valid_indices.append(idx)
        
        cleaned_df = df.loc[valid_indices].copy()
        return cleaned_df

    def format_part_number(self, val):
        """Helper to ensure part numbers are 4 digits, 0-padded strings."""
        try:
            # Handle floats (e.g. 123.0 -> "0123")
            s_val = str(int(float(val)))
            return s_val.zfill(4)
        except (ValueError, TypeError):
            return str(val).zfill(4)

    def run_logic(self, inputs: dict, output_folder: str, msg_queue: queue.Queue):
        """
        Executes the main business logic pipeline.

        Args:
            inputs: Dict containing paths for 'equip', 'parts', 'po'.
            output_folder: Path to save outputs.
            msg_queue: Thread-safe queue for GUI updates.
        """
        try:
            msg_queue.put(("info", "Starting processing..."))
            
            # ---------------------------------------------------------
            # 1. Load Reference Files
            # ---------------------------------------------------------
            msg_queue.put(("info", "Loading Master Files..."))
            
            # Load [01 Master File Part Numbers]
            df_parts = pd.read_excel(inputs['parts'])
            # Ensure strict types for merge keys
            df_parts['Equipment'] = df_parts['Equipment'].astype(str).str.strip()
            df_parts['Part No.'] = df_parts['Part No.'].apply(self.format_part_number)

            # Load [02 Master File PO Account Part Numbers]
            df_po = pd.read_csv(inputs['po']) if inputs['po'].lower().endswith('.csv') else pd.read_excel(inputs['po'])
            # Normalize column names for PO file if necessary (Assuming standard headers based on prompt logic)
            # Logic 3 implies columns: Account No., Part No., PO #, Quantity
            # If headers differ in reality, standardizing logic would go here.
            
            # ---------------------------------------------------------
            # 2. Logic Output 1: UCSD EQUIPMENT Equipment Import
            # ---------------------------------------------------------
            msg_queue.put(("info", "Processing Logic 1: Equipment Import..."))
            
            # Load Tab 0
            df_tab0 = self.load_messy_excel(inputs['equip'], sheet_index=0)
            
            # Keep required columns
            cols_req = ['ACCOUNT NO.', 'EQUIP', 'QTY']
            df_tab0 = df_tab0[cols_req].copy()
            
            # Merge Part No.
            df_tab0['EQUIP'] = df_tab0['EQUIP'].astype(str).str.strip()
            df_tab0 = pd.merge(df_tab0, df_parts[['Equipment', 'Part No.']], 
                               left_on='EQUIP', right_on='Equipment', how='left')
            
            # Identify Initial Uniques and Duplicates
            # Note: Logic implies we check uniqueness of ACCOUNT NO.
            counts = df_tab0['ACCOUNT NO.'].value_counts()
            
            df_unique_ids = counts[counts == 1].index
            df_dup_ids = counts[counts > 1].index
            
            df_01_unique = df_tab0[df_tab0['ACCOUNT NO.'].isin(df_unique_ids)].copy()
            df_02_duplicate = df_tab0[df_tab0['ACCOUNT NO.'].isin(df_dup_ids)].copy()
            
            # Remove specific items from duplicates
            filter_str = "H/C Filtration -Countertop Unit"
            df_02_duplicate = df_02_duplicate[df_02_duplicate['EQUIP'] != filter_str]
            
            # Re-evaluate uniqueness in the cleaned duplicate set
            # Some accounts might have had 2 rows, 1 was removed, now they are unique.
            counts_redux = df_02_duplicate['ACCOUNT NO.'].value_counts()
            new_unique_ids = counts_redux[counts_redux == 1].index
            
            # Move new uniques to df_01_unique
            rows_to_move = df_02_duplicate[df_02_duplicate['ACCOUNT NO.'].isin(new_unique_ids)]
            df_01_unique = pd.concat([df_01_unique, rows_to_move], ignore_index=True)
            
            # Remove moved rows from duplicate list
            df_02_duplicate = df_02_duplicate[~df_02_duplicate['ACCOUNT NO.'].isin(new_unique_ids)]
            
            # Format Output 1
            # Concatenate remaining duplicates (as per logic flow, though usually unique files only have uniques,
            # the prompt says "If they appear more than once, then keep them in 02_df_duplicate", 
            # implying Output 1 might be the 'Equipment Import' consisting of Valid Uniques?
            # *Clarification*: Prompt Output 1 doesn't explicitly say to drop the remaining duplicates from the output file,
            # but usually "Equipment Import" implies the clean list. 
            # However, looking at the file naming "UCSD EQUIPMENT Duplicate Import", that suggests Output 3/4 handles dups.
            # I will assume Output 1 is the Consolidated Clean List (Uniques).
            
            out1_df = df_01_unique.copy()
            out1_df['Part No.'] = out1_df['Part No.'].apply(self.format_part_number)
            
            # ---------------------------------------------------------
            # 3. Logic Output 2: UCSD EQUIPMENT STD Import
            # ---------------------------------------------------------
            msg_queue.put(("info", "Processing Logic 2: STD Import..."))
            
            df_tab2 = self.load_messy_excel(inputs['equip'], sheet_index=2)
            df_tab2 = df_tab2[['ACCOUNT NO.', 'EQUIP', 'QTY']].copy()
            
            # Merge Part No
            df_tab2['EQUIP'] = df_tab2['EQUIP'].astype(str).str.strip()
            df_tab2 = pd.merge(df_tab2, df_parts[['Equipment', 'Part No.']], 
                               left_on='EQUIP', right_on='Equipment', how='left')
            
            df_tab2['Part No.'] = df_tab2['Part No.'].apply(self.format_part_number)
            
            # Summarize (Group by Account, sum QTY)
            # Note: Logic asks to keep EQUIP in output. If we group by Account, EQUIP might be ambiguous if multiple exist.
            # Assuming grouping includes EQUIP to preserve it, or we take first.
            # Prompt: "Summarize the dataframe by Account No. and sum the QTY column. Ensure ... ACCOUNT NO., EQUIP, QTY, Part No."
            # If an account has multiple different EQUIPs, summing QTY requires grouping by all non-agg columns.
            out2_df = df_tab2.groupby(['ACCOUNT NO.', 'EQUIP', 'Part No.'], as_index=False)['QTY'].sum()
            
            # ---------------------------------------------------------
            # 4. Logic Output 3 & 4: Duplicates & Errors
            # ---------------------------------------------------------
            msg_queue.put(("info", "Processing Logic 3 & 4: Duplicates Checks..."))
            
            # Start with 02_df_duplicate from Logic 1
            dup_proc = df_02_duplicate[['ACCOUNT NO.', 'EQUIP', 'QTY', 'Part No.']].copy()
            
            # Summarize by Account and Part No
            dup_proc['Part No.'] = dup_proc['Part No.'].apply(self.format_part_number)
            
            # Sum QTY
            dup_grouped = dup_proc.groupby(['ACCOUNT NO.', 'Part No.'], as_index=False)['QTY'].sum()
            
            # Prepare Master PO for merge
            # Prompt implies Master PO has: Account No., Part No., PO #, Quantity
            # Let's standardize column names for the merge
            # Note: Mapping relies on user providing a file with recognizable headers.
            # We will attempt to map standard expected names.
            
            # Clean PO keys
            try:
                # Adjust these based on actual file headers
                # Looking for mapping: Account No, Part No
                po_acc_col = [c for c in df_po.columns if "Account" in c][0]
                po_part_col = [c for c in df_po.columns if "Part" in c][0]
                po_qty_col = [c for c in df_po.columns if "Quant" in c or "QTY" in c.upper()][0]
                po_num_col = [c for c in df_po.columns if "PO" in c][0]
            except IndexError:
                raise ValueError("Could not identify required columns (Account, Part, Quantity, PO) in PO Master File.")

            df_po['CleanAccount'] = df_po[po_acc_col]
            df_po['CleanPart'] = df_po[po_part_col].apply(self.format_part_number)
            
            # Left Merge
            merged_dups = pd.merge(dup_grouped, 
                                   df_po[[po_num_col, 'CleanAccount', 'CleanPart', po_qty_col]],
                                   left_on=['ACCOUNT NO.', 'Part No.'],
                                   right_on=['CleanAccount', 'CleanPart'],
                                   how='left')
            
            # Compare QTY vs Quantity
            # Handle NaNs in PO Quantity (if no match found) -> treat as 0 or Mismatch
            merged_dups[po_qty_col] = merged_dups[po_qty_col].fillna(0)
            
            def check_pass(row):
                try:
                    return "Pass" if float(row['QTY']) == float(row[po_qty_col]) else "Mismatch"
                except:
                    return "Mismatch"

            merged_dups['Status'] = merged_dups.apply(check_pass, axis=1)
            
            # Rename columns for final output
            # Target: PO #, Account #, Equipment, Quantity, Part #
            # We lost "Equipment" name in the groupby. We need to merge it back or just rename.
            # But wait, Groupby was on Account/Part. Equipment description might vary.
            # We will join back to df_parts to get the canonical Equipment Name for that Part No.
            merged_dups = pd.merge(merged_dups, df_parts[['Part No.', 'Equipment']], on='Part No.', how='left')

            final_cols_map = {
                po_num_col: 'PO #',
                'ACCOUNT NO.': 'Account #',
                'Equipment': 'Equipment',
                'QTY': 'Quantity', # Using the calculated QTY from input, or PO Quantity? Prompt says "Keep... Quantity". Usually matches PO.
                'Part No.': 'Part #'
            }
            
            # Output 3: Pass
            out3_df = merged_dups[merged_dups['Status'] == 'Pass'].copy()
            out3_df = out3_df.rename(columns=final_cols_map)
            out3_df = out3_df[list(final_cols_map.values())] # Order columns
            
            # Output 4: Mismatch
            # "Keep all columns"
            out4_df = merged_dups[merged_dups['Status'] == 'Mismatch'].copy()
            # We'll clean it up slightly for readability but keep all logic data
            
            # ---------------------------------------------------------
            # 5. Saving Files
            # ---------------------------------------------------------
            msg_queue.put(("info", "Saving Excel files..."))
            
            date_str = datetime.now().strftime("%Y%m%d")
            
            files_to_save = [
                (f"UCSD EQUIPMENT Equipment Import {date_str}.xlsx", out1_df),
                (f"UCSD EQUIPMENT STD Import {date_str}.xlsx", out2_df),
                (f"UCSD EQUIPMENT Duplicate Import {date_str}.xlsx", out3_df),
                (f"UCSD EQUIPMENT Duplicate ERRORS {date_str}.xlsx", out4_df)
            ]
            
            for fname, df in files_to_save:
                save_path = os.path.join(output_folder, fname)
                # Use XlsxWriter to force text format for Part Numbers
                with pd.ExcelWriter(save_path, engine='xlsxwriter') as writer:
                    df.to_excel(writer, index=False, sheet_name='Sheet1')
                    worksheet = writer.sheets['Sheet1']
                    # Auto-adjust columns (rough estimate)
                    for idx, col in enumerate(df.columns):
                        max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
                        worksheet.set_column(idx, idx, max_len)
            
            msg_queue.put(("success", "Processing Complete! Files saved."))

        except Exception as e:
            # Print stack trace to console for debugging, pass clean error to GUI
            import traceback
            traceback.print_exc()
            msg_queue.put(("error", f"Error: {str(e)}"))

# ==============================================================================
# GUI LAYER
# ==============================================================================

class ModernGUI(tk.Tk):
    def __init__(self):
        super().__init__()
        
        self.title("Automation Tool | Equipment Processor")
        self.geometry("700x550")
        self.configure(bg="#f0f2f5")
        
        # Data store for file paths
        self.file_paths = {
            'equip': None,
            'parts': None,
            'po': None,
            'output': None
        }
        
        # Threading communication
        self.msg_queue = queue.Queue()
        self.processor = DataProcessor()

        self._setup_styles()
        self._build_ui()

    def _setup_styles(self):
        style = ttk.Style()
        style.theme_use('clam')
        
        # Colors
        bg_color = "#f0f2f5"
        card_color = "#ffffff"
        accent_color = "#007acc"
        text_color = "#333333"
        
        # Frame Styles
        style.configure("Card.TFrame", background=card_color, relief="flat")
        style.configure("Main.TFrame", background=bg_color)
        
        # Label Styles
        style.configure("Header.TLabel", background=card_color, foreground=text_color, 
                        font=("Segoe UI", 16, "bold"))
        style.configure("Sub.TLabel", background=card_color, foreground="#666666", 
                        font=("Segoe UI", 10))
        style.configure("File.TLabel", background=card_color, foreground=text_color, 
                        font=("Segoe UI", 10))
        
        # Button Styles
        style.configure("Action.TButton", font=("Segoe UI", 9))
        
        style.configure("Run.TButton", background=accent_color, foreground="white", 
                        font=("Segoe UI", 11, "bold"), padding=10, borderwidth=0)
        style.map("Run.TButton", 
                  background=[('active', '#005f9e'), ('disabled', '#cccccc')])

    def _build_ui(self):
        # Main Container with padding
        main_container = ttk.Frame(self, style="Main.TFrame")
        main_container.pack(fill="both", expand=True, padx=20, pady=20)
        
        # Card (White Box)
        card = ttk.Frame(main_container, style="Card.TFrame")
        card.pack(fill="both", expand=True, padx=10, pady=10)
        
        # Internal Grid padding
        content_frame = ttk.Frame(card, style="Card.TFrame")
        content_frame.pack(fill="both", expand=True, padx=30, pady=30)
        
        # Header
        ttk.Label(content_frame, text="Equipment Data Processor", style="Header.TLabel").grid(row=0, column=0, columnspan=3, sticky="w", pady=(0, 5))
        ttk.Label(content_frame, text="Select input files to generate reports", style="Sub.TLabel").grid(row=1, column=0, columnspan=3, sticky="w", pady=(0, 20))
        
        # Rows
        self._create_file_row(content_frame, 2, "00 Equipment File", "equip")
        self._create_file_row(content_frame, 3, "01 Master File Part Numbers", "parts")
        self._create_file_row(content_frame, 4, "02 Master File PO Accounts", "po")
        
        ttk.Separator(content_frame, orient="horizontal").grid(row=5, column=0, columnspan=4, sticky="ew", pady=20)
        
        self._create_folder_row(content_frame, 6, "Output Folder", "output")
        
        # Status Log
        self.status_var = tk.StringVar(value="Ready")
        self.status_label = ttk.Label(content_frame, textvariable=self.status_var, 
                                      style="Sub.TLabel", foreground="#007acc")
        self.status_label.grid(row=7, column=0, columnspan=4, sticky="w", pady=(20, 5))
        
        # Run Button
        self.run_btn = ttk.Button(content_frame, text="RUN PROCESS", style="Run.TButton", 
                                  command=self.start_processing, state="disabled")
        self.run_btn.grid(row=8, column=0, columnspan=4, sticky="ew", pady=10)
        
        # Footer
        ttk.Label(content_frame, text="v1.0.0 | Segoe UI Interface", font=("Segoe UI", 8), foreground="#999999", background="#ffffff").grid(row=9, column=0, columnspan=4, pady=(20,0))

    def _create_file_row(self, parent, row, label_text, key):
        # Status Icon
        lbl_icon = ttk.Label(parent, text="○", font=("Segoe UI", 14), background="#ffffff", foreground="#cccccc")
        lbl_icon.grid(row=row, column=0, padx=(0, 10), pady=10)
        
        # Label
        ttk.Label(parent, text=label_text, style="File.TLabel", width=30).grid(row=row, column=1, sticky="w", pady=10)
        
        # Path Display (Truncated)
        lbl_path = ttk.Label(parent, text="No file selected", font=("Segoe UI", 9, "italic"), 
                             foreground="#999999", background="#ffffff", width=25)
        lbl_path.grid(row=row, column=2, sticky="w", padx=10, pady=10)
        
        # Button
        btn = ttk.Button(parent, text="Browse", style="Action.TButton", 
                         command=lambda: self.select_file(key, lbl_icon, lbl_path))
        btn.grid(row=row, column=3, padx=5, pady=10)

    def _create_folder_row(self, parent, row, label_text, key):
        lbl_icon = ttk.Label(parent, text="○", font=("Segoe UI", 14), background="#ffffff", foreground="#cccccc")
        lbl_icon.grid(row=row, column=0, padx=(0, 10), pady=10)
        
        ttk.Label(parent, text=label_text, style="File.TLabel", width=30).grid(row=row, column=1, sticky="w", pady=10)
        
        lbl_path = ttk.Label(parent, text="No folder selected", font=("Segoe UI", 9, "italic"), 
                             foreground="#999999", background="#ffffff", width=25)
        lbl_path.grid(row=row, column=2, sticky="w", padx=10, pady=10)
        
        btn = ttk.Button(parent, text="Select", style="Action.TButton", 
                         command=lambda: self.select_folder(key, lbl_icon, lbl_path))
        btn.grid(row=row, column=3, padx=5, pady=10)

    def select_file(self, key, icon_widget, path_widget):
        filetypes = [("Excel/CSV Files", "*.xlsx *.xls *.csv"), ("All Files", "*.*")]
        path = filedialog.askopenfilename(filetypes=filetypes)
        if path:
            self.file_paths[key] = path
            self.update_row_ui(icon_widget, path_widget, path)
            self.validate_inputs()

    def select_folder(self, key, icon_widget, path_widget):
        path = filedialog.askdirectory()
        if path:
            self.file_paths[key] = path
            self.update_row_ui(icon_widget, path_widget, path)
            self.validate_inputs()

    def update_row_ui(self, icon_widget, path_widget, path):
        # Update Icon
        icon_widget.config(text="✓", foreground="#28a745", font=("Segoe UI", 14, "bold"))
        # Update Path Text (Shorten if long)
        display_text = os.path.basename(path)
        if len(display_text) > 25:
            display_text = display_text[:22] + "..."
        path_widget.config(text=display_text, foreground="#333333", font=("Segoe UI", 9))

    def validate_inputs(self):
        # Check if all values in dictionary are not None
        if all(self.file_paths.values()):
            self.run_btn.config(state="normal")
        else:
            self.run_btn.config(state="disabled")

    def start_processing(self):
        self.run_btn.config(state="disabled")
        self.status_var.set("Initializing background process...")
        
        # Start Thread
        t = threading.Thread(target=self.processor.run_logic, 
                             args=(self.file_paths, self.file_paths['output'], self.msg_queue))
        t.daemon = True
        t.start()
        
        # Start checking queue
        self.check_queue()

    def check_queue(self):
        try:
            msg_type, content = self.msg_queue.get_nowait()
            
            if msg_type == "info":
                self.status_var.set(content)
                self.status_label.config(foreground="#007acc")
            elif msg_type == "success":
                self.status_var.set(content)
                self.status_label.config(foreground="#28a745")
                messagebox.showinfo("Success", content)
                self.run_btn.config(state="normal")
                return # Stop polling
            elif msg_type == "error":
                self.status_var.set("Error occurred.")
                self.status_label.config(foreground="#dc3545")
                messagebox.showerror("Error", content)
                self.run_btn.config(state="normal")
                return # Stop polling
                
        except queue.Empty:
            pass
        
        # Recursively call check_queue every 100ms
        self.after(100, self.check_queue)

# ==============================================================================
# ENTRY POINT
# ==============================================================================

def main():
    app = ModernGUI()
    app.mainloop()

if __name__ == "__main__":
    main()

c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
c:\Users\ipalacio\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")
Traceback (m